# CS2 (국내반) --- Lecture 12: Graph Shooter (NumPy, Matplotlib)
### Problem Set (in-class)

*이 노트북은 채점되지 않습니다. 수업 시간에 바로 이어서 풀면 됩니다.*

- 함수 그래프를 쏴서 장애물(회색 원)을 피해 과녁(빨간 원)을 맞히는 게임
- 발사대(검은 네모)는 항상 원점 (0, 0). 총알은 y = f(x)의 그래프를 따라 x = 0부터 과녁까지 날아감. 장애물에 닿으면 **BLOCKED**, 화면 밖(|y| > 10)이면 **OUT**, 과녁에 닿으면 **HIT**, 과녁 위치까지 가도 안 닿으면 **MISS**
- 위에서부터 **순서대로** 실행: 앞에서 만든 함수(무기, 충돌 판정, 엔진)를 뒤 문제에서 그대로 씀
- 빈칸 `___`을 채우거나 함수 몸통을 직접 작성 (4번은 실행만). 1~6번 필수, 7번 Optional

### Setup

아래 **게임 준비** 셀을 **먼저 한 번 실행**하세요. 게임의 레벨, 랜덤 맵, 그리기 함수가 준비됩니다. (무기와 엔진은 아래 문제에서 다룹니다.)

In [ ]:
#@title 게임 준비 (먼저 한 번 실행)
# L13_game.py : levels, random maps and drawing for Graph Shooter.
# The cannon is always at the origin (0, 0); the screen is 0 <= x <= 20, -10 <= y <= 10.
import numpy as np
import matplotlib.pyplot as plt

np.seterr(all="ignore")       # no warnings for huge values such as np.exp(100)

# walls:  obstacles (cx, cy, r)
# target: (tx, ty, r)
LEVELS = {
    1: dict(name="Straight Shot", walls=[], target=(18, 6, 0.7)),
    2: dict(name="Over the Hill", walls=[(10, -2, 4)], target=(18, 0, 0.7)),
    3: dict(name="Zigzag", walls=[(6, 3, 3), (13, -3, 3)], target=(19, 2, 0.7)),
    4: dict(name="The Wall", walls=[(10, -8, 2), (10, -4, 2), (10, 0, 2), (10, 4, 1.8)], target=(16, -6, 0.7)),
    5: dict(name="Step Up", walls=[(5, 5, 3.5), (14, -3, 3.5)], target=(18, 7, 0.7)),
    # bonus levels for the new weapons of P06
    6: dict(name="Valley", walls=[(6, 3, 4), (14, 3, 4), (10, -9.5, 2.5)], target=(18, -1, 0.7)),
    7: dict(name="Elevator", walls=[(5, 4, 3.5), (15, -4, 3.5)], target=(18, 6, 0.7)),
    8: dict(name="Ramp", walls=[(6, 3.5, 3), (13, -2, 2.5)], target=(18, 8, 0.7)),
}

def flightPath(f, level):
    """Where the bullet really flies (for drawing): from the origin until the first wall,
    until it leaves the screen, or until it reaches the target."""
    tx, ty, tr = level["target"]
    x = np.linspace(0, tx, 2001)
    y = f(x) + 0 * x
    y = y - y[0]
    stop = np.abs(y) > 10
    for cx, cy, r in level["walls"] + [level["target"]]:
        stop = stop | (np.sqrt((x - cx) ** 2 + (y - cy) ** 2) < r)
    end = np.argmax(stop) + 1 if np.any(stop) else len(x)
    return x[:end], y[:end]

def drawLevel(level, ax=None):
    if ax is None:
        plt.figure(figsize=(8, 8))
        ax = plt.gca()
    for cx, cy, r in level["walls"]:
        ax.add_patch(plt.Circle((cx, cy), r, color="gray"))
    tx, ty, tr = level["target"]
    ax.add_patch(plt.Circle((tx, ty), tr, color="red"))
    ax.plot(0, 0, "ks", markersize=12)
    ax.set_xlim(-0.6, 20)                         # a little room for the cannon
    ax.set_ylim(-10, 10)
    ax.set_aspect("equal")
    ax.set_xticks(range(0, 21))                   # a grid line at every integer
    ax.set_yticks(range(-10, 11))
    ax.tick_params(labelsize=7)
    ax.grid(True, color="0.85")
    ax.set_axisbelow(True)                        # grid lines behind the walls
    ax.set_title(level["name"])

def integerPoints(f, level):
    """The points of the shot at x = 0, 1, 2, ... (only where the bullet really flies)."""
    xs, ys = flightPath(f, level)
    x = np.arange(0, np.floor(xs[-1]) + 1)
    y = f(x) + 0 * x
    y = y - y[0]
    return x, y

def printPoints(x, y):
    """Print the points in rows:  x=0: 0.00   x=1: 0.33   ..."""
    row = ""
    for i in range(len(x)):
        row = row + ("x=" + str(int(x[i])) + ":").rjust(5) + ("%.2f" % y[i]).rjust(7) + "   "
        if i % 6 == 5 or i == len(x) - 1:
            print(row)
            row = ""

def drawShot(f, level, result):
    """Draw one level, the shot of f, and the result: drawShot(f, level, shoot(f, level)).
    The points at integer x are marked and printed, so you can see where to change f."""
    drawLevel(level)
    xs, ys = flightPath(f, level)
    plt.plot(xs, ys, "b", linewidth=1.5)
    xi, yi = integerPoints(f, level)
    plt.plot(xi, yi, "bo", markersize=4)
    plt.title(level["name"] + ": " + result)
    printPoints(xi, yi)
    plt.show()

# ---------------- random maps ----------------
def _canHit(level):
    """True if some simple shot (line, hill or step) hits the target. Used to skip impossible maps."""
    tx, ty, tr = level["target"]
    tries = []
    for a in np.linspace(-1, 1, 81):
        tries.append(lambda x, a=a: a * x)
    for h in np.linspace(-14, 14, 57):
        tries.append(lambda x, h=h: ty * x / tx - h * x * (x - tx) / 100)
        tries.append(lambda x, h=h: h / (1 + np.exp(-1.5 * (x - 10))))
    for f in tries:
        xs, ys = flightPath(f, level)
        if np.sqrt((xs[-1] - tx) ** 2 + (ys[-1] - ty) ** 2) < tr:
            return True
    return False

def randomLevel(seed):
    """A random map that can be cleared. The same seed always gives the same map."""
    rng = np.random.default_rng(seed)
    while True:
        walls = []
        for i in range(4):
            walls.append((rng.uniform(4, 15), rng.uniform(-8, 8), rng.uniform(1, 2.5)))
        target = (rng.uniform(16, 19), rng.uniform(-7, 7), 0.7)
        level = dict(name="Seed " + str(seed), walls=walls, target=target)
        free = True
        for cx, cy, r in walls:
            if np.sqrt((target[0] - cx) ** 2 + (target[1] - cy) ** 2) < r + 1.5:
                free = False
            if np.sqrt(cx ** 2 + cy ** 2) < r + 1:
                free = False
        xs, ys = flightPath(lambda x: 0 * x, level)          # a flat shot must not win
        flatHits = np.sqrt((xs[-1] - target[0]) ** 2 + (ys[-1] - target[1]) ** 2) < target[2]
        if free and not flatHits and _canHit(level):
            return level

def drawCatalog(functions):
    """functions: a dictionary {name: function}. Draws each one."""
    x = np.linspace(-4, 4, 801)
    plt.figure(figsize=(12, 6))
    k = 0
    for name in functions:
        k = k + 1
        plt.subplot(2, 3, k)
        plt.axhline(0, color="0.7")
        plt.axvline(0, color="0.7")
        plt.plot(x, functions[name](x), linewidth=2)
        plt.xlim(-4, 4)
        plt.ylim(-3, 4)
        plt.title(name)
    plt.tight_layout()
    plt.show()

# ---------------- AI demo ----------------
def showTraining(level, network, snapshots, losses):
    """snapshots: list of (step, p). Draws the shot after each step, then the loss curve."""
    plt.figure(figsize=(15, 3.6))
    for k in range(len(snapshots)):
        step, p = snapshots[k]
        ax = plt.subplot(1, len(snapshots), k + 1)
        drawLevel(level, ax)
        xs, ys = flightPath(lambda x: network(p, x), level)
        ax.plot(xs, ys, "b", linewidth=2)
        ax.set_xticks(range(0, 21, 5))
        ax.set_yticks(range(-10, 11, 5))
        ax.set_title("step " + str(step))
    plt.tight_layout()
    plt.show()
    plt.figure(figsize=(6, 3))
    plt.plot(losses)
    plt.yscale("log")
    plt.xlabel("step")
    plt.ylabel("loss")
    plt.title("loss: how bad the shot is")
    plt.grid(True)
    plt.show()

### Problem 1

(Plotting Practice) 함수 몸통을 직접 작성 (빈칸 없음).

1. Write a function `drawSquares`:
   - action: 점 (0, 0), (1, 1), (2, 4), (3, 9), (4, 16)을 좌표 list `x`, `y`로 만들고, 점과 선(`"o-"`)으로 그림. 제목 `y = x^2`
2. Write a function `drawSinCos`:
   - action: $0 \le x \le 2\pi$에서 100개의 점으로 $\sin x$와 $\cos x$를 label `sin`, `cos`와 함께 그림. 격자와 범례 표시

*$\pi$는 `np.pi`.*

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def drawSquares():
    # points (0, 0), (1, 1), (2, 4), (3, 9), (4, 16): y = x^2
    # 1. make the lists x and y
    # 2. plot them as dots joined by lines ("o-"), title "y = x^2", show
    pass

def drawSinCos():
    # 1. x: 100 numbers from 0 to 2 * np.pi    (np.linspace)
    # 2. plot sin x and cos x with labels "sin" and "cos"
    # 3. grid, legend, show
    pass

drawSquares()
drawSinCos()

### Problem 2

(Weapons from Deep Learning)

0. 맨 위에 import 두 줄을 직접 입력: `import numpy as np`, `import matplotlib.pyplot as plt`
1. Write functions `relu`, `leakyRelu`, `sigmoid`, `tanh`:
   - input parameter: an array `x`
   - return value: `x`의 각 원소에 대한 함수값의 array
     - ReLU: $x$와 $0$ 중 큰 값 (`np.maximum`)
     - Leaky ReLU: $x > 0$이면 $x$, 아니면 $0.1x$ (`np.where`)
     - sigmoid: $\dfrac{1}{1 + e^{-x}}$
     - tanh: $\dfrac{e^{x} - e^{-x}}{e^{x} + e^{-x}}$ (`np.tanh`를 쓰지 말고 공식으로)
2. Write a function `drawFunctions`:
   - action: $-5 \le x \le 5$에서 네 함수의 그래프를 `label`과 함께 그림. 보이는 범위는 x: $-5 \sim 5$, y: $-2 \sim 5$, 격자와 범례 표시
   - 함수 몸통 전체를 직접 작성 (`np.linspace`, `plt.plot`, `plt.xlim`, `plt.ylim`, `plt.grid`, `plt.legend`, `plt.show`)

*$e^{x}$는 `np.exp(x)`. 여기서 만든 함수가 Problem 5, 6의 **무기**가 됩니다.*

In [ ]:
# Type the two import lines: numpy as np, matplotlib.pyplot as plt

def relu(x):
    return ___

def leakyRelu(x):
    return ___

def sigmoid(x):
    return ___

def tanh(x):
    return ___

def drawFunctions():
    # 1. x: 201 numbers from -5 to 5          (np.linspace)
    # 2. plot relu, leakyRelu, sigmoid, tanh  (plt.plot with label=...)
    # 3. x axis -5 ~ 5, y axis -2 ~ 5, grid, legend, show
    pass

p = np.array([-2, 0, 3])
print(relu(p))                  # [0 0 3]
print(leakyRelu(p))             # [-0.2  0.   3. ]
print(np.round(sigmoid(p), 3))  # [0.119 0.5   0.953]
print(np.round(tanh(p), 3))     # [-0.964  0.     0.995]
drawFunctions()

### Problem 3

(Collision)

1. Write a function `insideCircle`:
   - input parameters: arrays `x`, `y`, numbers `cx`, `cy`, `r`
   - return value: 점 `(x[i], y[i])`가 중심 `(cx, cy)`, 반지름 `r`인 원 안에 있으면 True인 mask
2. Write a function `countInside`:
   - input parameters: arrays `x`, `y`, numbers `cx`, `cy`, `r`
   - return value: 원 안에 있는 점의 개수

*`np.sum(mask)`는 True의 개수. 테스트 코드의 그림: 원 안의 점(`inside`가 True)은 빨간색, 나머지는 파란색 (그리기 코드는 주어짐).*

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def insideCircle(x, y, cx, cy, r):
    return ___

def countInside(x, y, cx, cy, r):
    return ___

x = np.array([0, 1, 2, 3, 4])
y = np.array([0, 0, 0, 0, 0])
print(insideCircle(x, y, 3, 0, 1.5))         # [False False  True  True  True]
print(np.any(insideCircle(x, y, 10, 0, 1)))  # False
print(countInside(x, y, 3, 0, 1.5))          # 3

# 500 random points and a circle (center (1, 1), radius 3)
rng = np.random.default_rng(13)
px = rng.uniform(-5, 5, 500)
py = rng.uniform(-5, 5, 500)
print(countInside(px, py, 1, 1, 3))          # 142

inside = insideCircle(px, py, 1, 1, 3)     # True: in the circle
plt.plot(px[inside], py[inside], "r.")     # in the circle: red
plt.plot(px[inside == False], py[inside == False], "b.")   # others: blue
plt.gca().set_aspect("equal")
plt.show()

### Problem 4

(The Shooting Engine) 게임 엔진은 **완성되어 있음** (빈칸 없음). 코드를 읽고 실행해서 결과 네 가지를 확인.
- `startAtOrigin(y)`: 그래프를 위아래로 옮겨서 원점 (0, 0)에서 출발
- `offScreen(y)`: |y| > 10인 값이 하나라도 있으면 True
- `touches(x, y, circle)`: 총알의 점 중 하나라도 원 안에 있으면 True (Problem 3의 `insideCircle` 이용)
- `shoot(f, level)`: 화면 밖이면 `"OUT"`, 장애물에 닿으면 `"BLOCKED"`, 과녁에 닿으면 `"HIT"`, 그 외 `"MISS"`

*테스트의 `flat`, `hill`, `down`, `steep`이 각각 어떤 결과가 나오는지 그림과 함께 확인.*

In [ ]:
import numpy as np

def startAtOrigin(y):            # move the graph up or down: y[0] becomes 0
    return y - y[0]

def offScreen(y):                # did it leave the screen (|y| > 10)?
    return np.any(np.abs(y) > 10)

def touches(x, y, circle):       # did the bullet touch this circle?
    cx, cy, r = circle
    return np.any(insideCircle(x, y, cx, cy, r))

# the engine: read it like a sentence
def shoot(f, level):
    tx, ty, tr = level["target"]
    x = np.linspace(0, tx, 2001)         # from x = 0 to the target
    y = startAtOrigin(f(x))
    if offScreen(y):
        return "OUT"
    for wall in level["walls"]:
        if touches(x, y, wall):
            return "BLOCKED"
    if touches(x, y, level["target"]):
        return "HIT"
    return "MISS"

xs = np.array([0, 1, 2])
ys = np.array([-4, 1, 6])
print(startAtOrigin(ys))                # [ 0  5 10]
print(offScreen(np.array([1, 12, 3])))  # True
print(touches(xs, ys, (2, 6, 0.5)))     # True

def flat(x):
    return 0 * x

def hill(x):
    return -x * (x - 18) / 20

def down(x):
    return -0.3 * x

def steep(x):
    return 0.6 * x

level = dict(name="Test", walls=[(10, 0, 2)], target=(18, 0, 0.7))
print(shoot(flat, level))               # BLOCKED
print(shoot(hill, level))               # HIT
print(shoot(down, level))               # MISS
print(shoot(steep, level))              # OUT
drawShot(hill, level, shoot(hill, level))

### Problem 5

(Mission: Clear the Levels)

1. Write a function `aim` (조준경):
   - input parameter: a function `f`
   - action: 장애물 없이 내 함수의 그래프만 그림. $0 \le x \le 20$에서 $y = f(x) - f(0)$ (원점에서 출발), 보이는 범위 x: 0 ~ 20, y: -10 ~ 10, 격자, 제목 `aim`
   - 함수 몸통 전체를 직접 작성 (Problem 2의 `drawFunctions`와 같은 방법)
2. 레벨 1~5를 **한 판씩** 깬다. 레벨마다 함수 `f1` ~ `f5`를 고쳐서 과녁을 맞히기. 쏘기 전에 `aim`으로 먼저 모양을 확인. 과녁에 맞기만 하면 어떤 함수든 정답.
- 처음에는 `return 0 * x`: 그대로 실행해도 됨. 결과와 그림을 보고 고쳐 가면서 깨기
- 그림의 파란 점: x = 0, 1, 2, ... (정수)에서 총알의 위치. 그 값(y)도 함께 출력되므로 숫자를 보고 고치기
- 무기: Problem 2에서 만든 `relu`, `leakyRelu`, `sigmoid`, `tanh`와 `np.sin`, `np.cos`, `np.abs`, ... 도우미 함수를 `def`로 더 만들어도 됨
- 엔진: Problem 4에서 만든 `shoot`. 출력이 `HIT`이면 그 레벨 클리어

| 식 | 모양 |
|---|---|
| `a * x` | 직선, `a`가 클수록 가파름 |
| `-x * (x - 18)` | 원점과 x = 18을 지나는 산 모양 |
| `a * np.sin(b * x)` | 물결: `a`는 높이, `b`가 클수록 촘촘 |
| `a * relu(x - c)` | x = c까지 평평, 그다음 직선 |
| `a * sigmoid(x - c)` | x = c 근처에서 계단 오르기 (높이 a) |
| `a * tanh(x - c)` | x = c 근처에서 -a에서 a로 |
| `f(x - c)` | 오른쪽으로 c만큼 이동 |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Your scope: draw only the graph of f (no walls) before you shoot.
def aim(f):
    # 1. x: 201 numbers from 0 to 20
    # 2. plot y = f(x) - f(0)   (the graph starts at the origin)
    # 3. x axis 0 ~ 20, y axis -10 ~ 10, grid, title "aim", show
    pass

# Weapons: relu, leakyRelu, sigmoid, tanh (P02), np.sin, np.cos, np.abs, ...
# Any function that hits the target is a correct answer!

#### Level 1: Straight Shot

In [ ]:
def f1(x):
    return 0 * x                     # change this!

aim(f1)                              # look through your scope first
result = shoot(f1, LEVELS[1])
print(result)                        # goal: HIT
drawShot(f1, LEVELS[1], result)

#### Level 2: Over the Hill

In [ ]:
def f2(x):
    return 0 * x                     # change this!

aim(f2)
result = shoot(f2, LEVELS[2])
print(result)                        # goal: HIT
drawShot(f2, LEVELS[2], result)

#### Level 3: Zigzag

In [ ]:
def f3(x):
    return 0 * x                     # change this!

aim(f3)
result = shoot(f3, LEVELS[3])
print(result)                        # goal: HIT
drawShot(f3, LEVELS[3], result)

#### Level 4: The Wall

In [ ]:
def f4(x):
    return 0 * x                     # change this!

aim(f4)
result = shoot(f4, LEVELS[4])
print(result)                        # goal: HIT
drawShot(f4, LEVELS[4], result)

#### Level 5: Step Up

In [ ]:
def f5(x):
    return 0 * x                     # change this!

aim(f5)
result = shoot(f5, LEVELS[5])
print(result)                        # goal: HIT
drawShot(f5, LEVELS[5], result)

### Problem 6

(Random Maps) `randomLevel(SEED)`: `SEED`에 넣은 숫자로 랜덤 맵을 만든다. 같은 숫자면 항상 같은 맵이고, 깰 수 있는 맵만 나온다.
- `myShot`을 고쳐 가며 맵을 깨기 (처음에는 `return 0 * x`)
- 깨면 `SEED`를 바꿔서 다음 맵. 여러 SEED에 도전

In [ ]:
import numpy as np

SEED = 4                # any number: a new map (1, 2, 3, ..., 2027)
level = randomLevel(SEED)

def myShot(x):
    return 0 * x                     # change this!

result = shoot(myShot, level)
print(result)                        # goal: HIT
drawShot(myShot, level, result)

### Optional Problem (Problem 7)

필수 문제와 달리 제출/검사 대상은 아니지만, 시간이 남으면 시도해보는 것을 권합니다.

(More Weapons and Bonus Levels)

1. Write functions `step`, `elu`, `softplus`, `swish`, `myAbs`:
   - step: $x > 0$이면 $1$, 아니면 $0$
   - ELU: $x > 0$이면 $x$, 아니면 $e^{x} - 1$
   - softplus: $\log(1 + e^{x})$ (`np.log`)
   - swish: $x \times \text{sigmoid}(x)$ (Problem 2의 `sigmoid` 이용)
   - myAbs: $|x|$ (`np.abs`를 쓰지 말고 `np.where`로)
2. 새 무기로 보너스 레벨 6~8을 깨기: `f6`, `f7`, `f8` (처음에는 `return 0 * x`)
   - Level 6 Valley: V자로 내려갔다 올라가기
   - Level 7 Elevator: 수직으로 점프
   - Level 8 Ramp: 평평하다가 올라가기

In [ ]:
import numpy as np

def step(x):
    return ___

def elu(x):
    return ___

def softplus(x):
    return ___

def swish(x):
    return ___

def myAbs(x):
    return ___

p = np.array([-2, 0, 3])
print(step(p))                   # [0 0 1]
print(np.round(elu(p), 3))       # [-0.865  0.     3.   ]
print(np.round(softplus(p), 3))  # [0.127 0.693 3.049]
print(np.round(swish(p), 3))     # [-0.238  0.     2.858]
print(myAbs(p))                  # [2 0 3]
drawCatalog({"step": step, "elu": elu, "softplus": softplus,
             "swish": swish, "myAbs": myAbs, "sigmoid": sigmoid})

#### Level 6: Valley

In [ ]:
def f6(x):
    return 0 * x                     # change this!

result = shoot(f6, LEVELS[6])
print(result)                        # goal: HIT
drawShot(f6, LEVELS[6], result)

#### Level 7: Elevator

In [ ]:
def f7(x):
    return 0 * x                     # change this!

result = shoot(f7, LEVELS[7])
print(result)                        # goal: HIT
drawShot(f7, LEVELS[7], result)

#### Level 8: Ramp

In [ ]:
def f8(x):
    return 0 * x                     # change this!

result = shoot(f8, LEVELS[8])
print(result)                        # goal: HIT
drawShot(f8, LEVELS[8], result)

### Bonus: AI가 대신 쏘기 (실행만)

지금까지 여러분은 `3 * relu(x - 4) + 6 * sigmoid(x - 15)`처럼 **무기를 크기만큼 곱하고, 옮기고, 더해서** 총알을 만들었습니다. 이 모양이 바로 **신경망(neural network)** 입니다.

- `network(p, x)`: sigmoid 뉴런 6개의 합. 숫자 `p`(각 뉴런의 크기 `a`, 기울기 `w`, 위치 `c`)가 총알 모양을 정함
- `loss(p, level)`: 총알이 얼마나 나쁜지 (과녁에서 빗나간 거리 + 장애물에 들어간 정도 + 화면 밖으로 나간 정도). 0이면 완벽
- `train(level)`: 숫자 하나하나를 아주 조금 바꿔 보고, loss가 줄어드는 쪽으로 모든 숫자를 조금씩 옮기기를 800번 반복 (**경사하강법**). 여러분이 손으로 하던 "고치고 다시 쏘기"를 컴퓨터가 대신 함

해 볼 것:
- `level = LEVELS[2]`, `randomLevel(7)` 등 다른 맵으로 바꿔 보기
- `LEVELS[4]` (The Wall): AI가 깰 수 있을까? 못 깬다면 왜 그럴까?
- `N`(뉴런 개수)을 2, 10으로 바꿔 보기

In [ ]:
import numpy as np

N = 6       # number of neurons

# a tiny neural network: the sum of N sigmoid neurons
# p = [a_1 .. a_N, w_1 .. w_N, c_1 .. c_N]
def network(p, x):
    a, w, c = p[:N], p[N:2 * N], p[2 * N:]
    y = 0 * x
    for i in range(N):
        y = y + a[i] * sigmoid(w[i] * (x - c[i]))
    return y

# loss: how bad the shot is (0 = perfect)
def loss(p, level):
    tx, ty, tr = level["target"]
    x = np.linspace(0, tx, 201)
    y = network(p, x)
    y = y - y[0]
    L = (y[-1] - ty) ** 2                       # missed the target
    for cx, cy, r in level["walls"]:
        d = np.sqrt((x - cx) ** 2 + (y - cy) ** 2)
        L = L + 300 * np.mean(np.maximum(0, r + 0.6 - d) ** 2)  # in a wall
    L = L + 10 * np.mean(np.maximum(0, np.abs(y) - 9.5) ** 2)   # off screen
    return L

# training: change every number a tiny bit, see if the loss goes down,
# and move all numbers a little in the direction that lowers the loss
def train(level, steps=800, lr=0.005, seed=0):
    rng = np.random.default_rng(seed)
    a = rng.normal(0, 0.3, N)          # almost flat at first
    w = np.ones(N)
    c = np.linspace(2, 18, N)          # neurons spread over the screen
    p = np.concatenate([a, w, c])
    snapshots = []
    losses = []
    for step in range(steps + 1):
        if step in [0, 5, 20, 80, steps]:
            snapshots.append((step, p.copy()))
        L = loss(p, level)
        losses.append(L)
        grad = np.zeros(len(p))
        for j in range(len(p)):
            q = p.copy()
            q[j] = q[j] + 0.0001
            grad[j] = (loss(q, level) - L) / 0.0001
        p = p - lr * np.clip(grad, -20, 20)
    return p, snapshots, losses

level = LEVELS[5]           # try LEVELS[1] ~ LEVELS[8], randomLevel(7)
p, snapshots, losses = train(level)
showTraining(level, network, snapshots, losses)

def aiShot(x):
    return network(p, x)

result = shoot(aiShot, level)
print(result)  # HIT
drawShot(aiShot, level, result)